# 24/365 화염·연기 탐지 후속 실험

**질문** · 학습한 모델은 어떤 장면의 연기를 놓치며, confidence를 낮추면 어떤 오탐이 생기는가?

2025년 팀 프로젝트에서 제안한 카메라 기반 탐지 방식을 데이터와 결과로 다시 검증한다. 이 노트북은 저장소의 학습·평가 명령을 실행하고 결과를 읽는다. 학습과 평가는 해당 셀에서 실행 변수를 켠 뒤 진행한다.

In [ ]:
import json
import shlex
import subprocess
import sys
from pathlib import Path
from random import Random

import torch
from IPython.display import Image as DisplayImage, Markdown, display
from PIL import Image, ImageDraw

here = Path.cwd().resolve()
repo = next((path for path in (here, *here.parents) if (path / "wildfire").is_dir()
             and (path / "data.example.yaml").is_file()), None)
if repo is None:
    raise FileNotFoundError("이 저장소 또는 notebooks 폴더에서 커널을 시작하세요.")
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))

from wildfire.dataset import load_dataset, sha256
from wildfire.evaluation import target_boxes

DATA = repo / "data.yaml"
DATASET_VERSION = "roboflow-v2"
RUNS = repo / "runs"
DEVICE = "0"  # NVIDIA의 첫 GPU. 맥북에서 짧게 확인할 때는 "cpu".
MODEL = "yolo11s.pt"
IMAGE_SIZE = 640
BATCH = 8
PILOT_EPOCHS = 3
TRAIN_EPOCHS = 50
PILOT_RUN = "notebook-yolo11s-pilot"
TRAIN_RUN = "notebook-yolo11s-baseline"
WEIGHTS = RUNS / TRAIN_RUN / "train" / "weights" / "best.pt"
COMMON = [
    "--data", str(DATA), "--dataset-version", DATASET_VERSION,
    "--device", DEVICE, "--imgsz", str(IMAGE_SIZE), "--batch", str(BATCH),
    "--runs-dir", str(RUNS),
]

def run_workflow(command, *arguments):
    process_args = [sys.executable, "-m", "wildfire", command, *COMMON, *arguments]
    print(shlex.join(process_args), flush=True)
    with subprocess.Popen(
        process_args, cwd=repo, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT
    ) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
        if process.wait():
            raise subprocess.CalledProcessError(process.returncode, process_args)

print(f"Python: {sys.executable}")
print(f"Device: {DEVICE} | CUDA available: {torch.cuda.is_available()}")
if DEVICE.isdigit() and int(DEVICE) >= torch.cuda.device_count():
    print("이 커널에서 지정한 CUDA 장치를 사용할 수 없습니다. 학습 전에 커널과 DEVICE를 확인하세요.")

## 1. 클래스와 라벨 확인

실제 export의 클래스 ID와 설정을 대조한다. 예시 박스와 이름이 일치하는지 직접 확인한다. 데이터 점검은 분할 간 바이트가 같은 이미지를 잡으며, 같은 영상의 유사 프레임은 별도로 살펴본다.

In [ ]:
if not DATA.is_file():
    raise FileNotFoundError("README를 따라 data.example.yaml을 복사하고 로컬 data.yaml을 수정하세요.")

dataset = load_dataset(DATA)
dataset_fingerprint = dataset.manifest()["sha256"]
print(json.dumps(dataset.counts, indent=2, ensure_ascii=False))
print(f"Class IDs: {dataset.names}")
print(f"Dataset SHA-256: {dataset_fingerprint}")

records = dataset.images["train"]
for record in Random(0).sample(records, min(4, len(records))):
    with Image.open(dataset.root / record.image) as original:
        canvas = original.convert("RGB")
    canvas.thumbnail((560, 400))
    draw = ImageDraw.Draw(canvas)
    for box in target_boxes(dataset.root / record.label, dataset.names):
        x1, y1, x2, y2 = box.xyxy
        pixels = (x1 * canvas.width, y1 * canvas.height, x2 * canvas.width, y2 * canvas.height)
        draw.rectangle(pixels, outline="#4ade80", width=3)
        draw.text(pixels[:2], dataset.names[box.class_id], fill="#4ade80", stroke_width=1, stroke_fill="black")
    print(record.image)
    display(canvas)

## 2. 시험 학습과 본 학습

먼저 짧은 실행의 클래스, 박스, 메모리, 두 번째·세 번째 epoch 시간을 확인한다. 그 결과를 보고 본 학습의 batch와 epoch를 결정한다. 실패 후 같은 이름으로 재시도하려면 새 실행 이름을 지정한다.

In [ ]:
RUN_PILOT = False

if RUN_PILOT:
    run_workflow(
        "train", "--model", MODEL, "--epochs", str(PILOT_EPOCHS), "--name", PILOT_RUN,
    )
else:
    print("시험 학습을 실행하려면 RUN_PILOT을 True로 바꾸세요.")

In [ ]:
RUN_TRAIN = False

if RUN_TRAIN:
    run_workflow(
        "train", "--model", MODEL, "--epochs", str(TRAIN_EPOCHS), "--name", TRAIN_RUN,
    )
else:
    print(f"본 학습 결과를 사용할 경로: {WEIGHTS}")

## 3. Validation에서 confidence 선택

후보 confidence를 바꿔 같은 가중치와 validation split에 적용한다. Smoke Recall이 얼마나 달라지고 어떤 FP가 늘었는지 비교한 뒤 운영값을 정한다. 점수가 정의되지 않는 경우는 표에서 대시로 표시한다.

In [ ]:
def evaluation_name(split, confidence):
    return f"{TRAIN_RUN}-{split}-c{confidence:g}"

def evaluate(split, confidence):
    run_workflow(
        "evaluate", "--weights", str(WEIGHTS), "--split", split,
        "--confidence", str(confidence), "--name", evaluation_name(split, confidence),
    )

def read_report(split, confidence):
    run_dir = RUNS / evaluation_name(split, confidence)
    metrics_path = run_dir / "metrics.json"
    if not metrics_path.is_file():
        return run_dir, None

    evidence = json.loads((run_dir / "run.json").read_text(encoding="utf-8"))
    report = json.loads(metrics_path.read_text(encoding="utf-8"))
    if evidence["dataset_sha256"] != dataset_fingerprint:
        raise ValueError("저장된 평가의 데이터가 현재 설정과 다릅니다.")
    if evidence["input_checkpoint_sha256"] != sha256(WEIGHTS):
        raise ValueError("저장된 평가의 가중치가 현재 가중치와 다릅니다.")
    if (report["split"], report["operating_point"]["confidence_min"]) != (split, confidence):
        raise ValueError("저장된 평가의 split 또는 confidence가 선택값과 다릅니다.")
    return run_dir, report

def metric_text(value):
    return "—" if value is None else f"{value:.3f}"

def show_report(split, confidence, examples=3):
    run_dir, report = read_report(split, confidence)
    if report is None:
        print(f"평가 결과를 기다리는 경로: {run_dir}")
        return
    print(f"{split} · confidence {confidence:g} · {report['counts']['images']} images")
    rows = [
        "| 클래스 | 정답 수 | AP50 | AP50–95 | TP | FP | FN | Precision | Recall |",
        "|---|---:|---:|---:|---:|---:|---:|---:|---:|",
    ]
    for name, values in report["operating_point"]["classes"].items():
        ap = report["ap"]["classes"][name] or {}
        count = report["counts"]["instances"][name]
        rows.append(
            f"| {name} | {count} | {metric_text(ap.get('ap50'))} | {metric_text(ap.get('ap50_95'))}"
            f" | {values['tp']} | {values['fp']} | {values['fn']} | {metric_text(values['precision'])}"
            f" | {metric_text(values['recall'])} |"
        )
    display(Markdown("\n".join(rows)))
    for image in sorted((run_dir / "examples").glob("*.jpg"))[:examples]:
        print(image.name)
        display(DisplayImage(filename=str(image), width=560))

In [ ]:
RUN_VALIDATION = False
VALIDATION_CONFIDENCE = 0.25

if RUN_VALIDATION:
    evaluate("val", VALIDATION_CONFIDENCE)
show_report("val", VALIDATION_CONFIDENCE)

## 4. 설정을 고정한 뒤 Test 평가

선택한 confidence와 근거를 작성하고 해당 설정의 validation 평가 기록을 확인한 뒤 test를 한 번 실행한다. Test의 오류 이미지는 FP 예측을 빨간색, FN 정답을 노란색으로 표시한다. 이미지의 파일명과 정답 라벨은 평가 폴더의 errors.jsonl에서 확인할 수 있다.

In [ ]:
RUN_TEST = False
SELECTED_CONFIDENCE = None
SELECTION_REASON = ""

if SELECTED_CONFIDENCE is None:
    print("validation 비교 후 SELECTED_CONFIDENCE와 SELECTION_REASON을 채우세요.")
else:
    val_dir, validation_report = read_report("val", SELECTED_CONFIDENCE)
    if validation_report is None:
        raise ValueError(f"선택한 confidence의 validation 결과가 필요합니다: {val_dir}")
    if not SELECTION_REASON.strip():
        raise ValueError("SELECTION_REASON에 선택 근거를 작성하세요.")
    print(f"선택 근거: {SELECTION_REASON}")
    if RUN_TEST:
        evaluate("test", SELECTED_CONFIDENCE)
        test_dir = RUNS / evaluation_name("test", SELECTED_CONFIDENCE)
        (test_dir / "threshold-decision.json").write_text(
            json.dumps({
                "confidence": SELECTED_CONFIDENCE,
                "reason": SELECTION_REASON,
                "validation_run": val_dir.name,
            }, ensure_ascii=False, indent=2) + "\n",
            encoding="utf-8",
        )
    show_report("test", SELECTED_CONFIDENCE)

## 5. 포트폴리오에 가져갈 근거

- Validation에서 어떤 오탐·미탐을 보고 confidence를 선택했는가?
- Test에서 놓친 연기 사례 하나와 오탐 사례 하나를 골라 **관찰 → 가능한 원인 → 다음 검증 방법**을 기록한다. 오탐 예시가 없다면 해당 표본과 임계값에서 관찰된 범위를 설명한다.
- 공개하는 지표는 학습·평가 폴더의 run.json과 dataset.json에 연결한다. 이미지 단위 결과와 현장 경보 효과를 구분한다.
- 저장소에 올릴 때는 셀 출력을 지운다. 원본 데이터나 이미지 공유가 허용되면 별도 증거 자료에서 필요한 사례만 인용한다.